# Name the categories from a fictional taxonomy

Issue: #9

## Question

The classifier (#34) and the judge (#39) need readable category names, and our clusters only have IDs. How do we name them so that related clusters get related names?

## Data used

Inputs: `artifacts/taxonomy.json` (a fixed, fictional fashion-and-home tree with 80 leaves), `data/categories/centroids.npy` and `item_category.parquet` (#8).

Output: `artifacts/category_names.json` (cluster ID to name, with `synthetic: true`).

**Where the taxonomy came from.** No API key was used. The taxonomy was written directly by Claude (claude-sonnet-5) in the Claude Code session on 2026-09-19 and saved once as a fixed file. This notebook only reads it, so it never calls an LLM and cannot change between runs. It is fictional and not derived from OTTO.

**All names are synthetic.** They carry no real meaning about the items. They only make the clusters readable for the classifier and the judge.

## How clusters are paired with names

1. Build a hierarchy over the 80 cluster centroids (average linkage, cosine distance). Its leaf order puts similar clusters next to each other.
2. Order the taxonomy leaves by their position in the tree (depth-first), so leaves under the same group sit next to each other.
3. Pair the two orders one to one. Clusters that are close in behaviour therefore get names from the same group.

The pairing is only as good as the match between the two orderings. It has no true semantics, so a cluster named `Dresses` is not really about dresses.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import polars as pl
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform

DATA = Path("data") if Path("data").exists() else Path("../data")
ART = Path("artifacts") if Path("artifacts").exists() else Path("../artifacts")

tree = json.loads((ART / "taxonomy.json").read_text())
centroids = np.load(DATA / "categories" / "centroids.npy")
categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
K = centroids.shape[0]

# leaves in tree order (depth-first), each with its group
leaves = [(leaf, group["name"]) for group in tree["children"] for leaf in group["children"]]
print(f"{K} clusters, {len(leaves)} taxonomy leaves in {len(tree['children'])} groups")
assert K == len(leaves) == len({leaf for leaf, _ in leaves})

In [ ]:
# 1. order the clusters by hierarchical clustering of their centroids
link = linkage(centroids, method="average", metric="cosine")
order = dendrogram(link, no_plot=True)["leaves"]          # cluster IDs, similar ones adjacent
assert sorted(order) == list(range(K))

# 2 and 3. pair the ordered clusters with the tree-ordered leaves
names = {}
for cluster, (leaf, group) in zip(order, leaves, strict=True):
    names[int(cluster)] = {"name": leaf, "group": group, "synthetic": True}

out = {
    "synthetic": True,
    "note": "Names are fictional and carry no true meaning about the items.",
    "taxonomy": "artifacts/taxonomy.json",
    "method": "average-linkage dendrogram order of centroids paired with depth-first taxonomy leaf order",
    "categories": {str(c): names[c] for c in sorted(names)},
}
(ART / "category_names.json").write_text(json.dumps(out, indent=2, ensure_ascii=False))
print("written artifacts/category_names.json")

## Checks

In [ ]:
saved = json.loads((ART / "category_names.json").read_text())
cats = saved["categories"]
assert len(cats) == K
assert len({v["name"] for v in cats.values()}) == K, "duplicate names"
assert all(v["synthetic"] for v in cats.values()) and saved["synthetic"]
assert set(map(int, cats)) == set(categories["cluster"].unique().to_list())
print(f"{len(cats)} names, all unique, all marked synthetic, IDs match the clusters in item_category.parquet")

## Do close clusters get names from the same group?

For each cluster, take its nearest other cluster by centroid cosine similarity. If the pairing worked, the two names should come from the same group (siblings) more often than by chance.

In [ ]:
dist = squareform(pdist(centroids, metric="cosine"))
np.fill_diagonal(dist, np.inf)
nearest = dist.argmin(axis=1)
group_of = {int(c): v["group"] for c, v in cats.items()}
name_of = {int(c): v["name"] for c, v in cats.items()}
same = [group_of[c] == group_of[int(nearest[c])] for c in range(K)]
observed = float(np.mean(same))

# chance: probability that two random different leaves share a group
sizes = np.array([len(g["children"]) for g in tree["children"]])
chance = float((sizes * (sizes - 1)).sum() / (K * (K - 1)))
print(f"nearest cluster is in the same group: {observed:.0%} of clusters (chance {chance:.0%})")

mutual = [(c, int(nearest[c])) for c in range(K) if int(nearest[nearest[c]]) == c and c < int(nearest[c])]
sibling_examples = [(a, b) for a, b in mutual if group_of[a] == group_of[b]]
print("mutual nearest pairs:", len(mutual), " of which siblings under one group:", len(sibling_examples))
for a, b in sibling_examples[:5]:
    print(f"  cluster {a} '{name_of[a]}' <-> cluster {b} '{name_of[b]}'  (both under {group_of[a]}, synthetic names)")
assert len(sibling_examples) >= 3, "fewer than 3 examples of close clusters with sibling names"
assert observed > chance

## Five clusters with their (synthetic) names and most popular items

In [ ]:
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from split import WINDOWS, day_to_ms

TRAIN_END = day_to_ms(WINDOWS["w0 initial (weeks 1-2)"]["train_end_day"])
pop = (
    pl.scan_parquet(DATA / "sample" / "events.parquet")
    .filter((pl.col("type") == 0) & (pl.col("ts") < TRAIN_END))
    .group_by("aid").agg(pl.len().alias("clicks")).collect()
)
top = (
    categories.join(pop, on="aid", how="left").with_columns(pl.col("clicks").fill_null(0))
    .sort(["cluster", "clicks", "aid"], descending=[False, True, False])
    .group_by("cluster", maintain_order=True).agg(pl.col("aid").head(5).alias("top_items"), pl.len().alias("items"))
)
sample = top.filter(pl.col("cluster").is_in([0, 12, 29, 49, 62])).with_columns(
    pl.col("cluster").map_elements(lambda c: name_of[c], return_dtype=pl.Utf8).alias("synthetic_name"),
    pl.col("cluster").map_elements(lambda c: group_of[c], return_dtype=pl.Utf8).alias("synthetic_group"),
).select("cluster", "synthetic_name", "synthetic_group", "items", "top_items")
with pl.Config(fmt_str_lengths=80, tbl_width_chars=160):
    print(sample)

## Result

All 80 clusters have a unique synthetic name, saved in `artifacts/category_names.json`. Names of close clusters come from the same taxonomy group more often than by chance (see the printout), so related clusters have related names. The names are labels only: they say nothing true about the items.

## What I learned

To be written by the owner of the project.